In [ ]:
import numpy as np
import xarray as xr
import datetime as date
import pandas as pd
from datetime import datetime
from datetime import timedelta
import datetime as dt
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import erddapy
from erddapy import ERDDAP
from cmocean import cm as cmo
from pathlib import Path
import os
import glob
import shutil
from sklearn.linear_model import LinearRegression
myFmtlong = mdates.DateFormatter('%m/%d\n%H:%M')
myFmt = mdates.DateFormatter('%m/%d')

## Set plotting parameters
SMALL_SIZE = 12
MEDIUM_SIZE = 15
BIGGER_SIZE = 20

# increase text sizes because the figure is so big
fac =1.5
plt.rc('font', size=SMALL_SIZE * fac)          # controls default text sizes
plt.rc('axes', titlesize=MEDIUM_SIZE * fac)     # fontsize of the axes title
plt.rc('axes', labelsize=MEDIUM_SIZE * fac)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE * fac)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE * fac)    # fontsize of the tick labels
plt.rc('legend', fontsize=SMALL_SIZE * fac)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE * fac *0.8)  # fontsize of the figure title

In [ ]:
## Paths -- edit these to match your local setup
DATA_DIR = './data'
FORCING_DIR = f'{DATA_DIR}/forcing'
OUTPUT_DIR = f'{DATA_DIR}/final_run/data'
FIGURES_DIR = f'{DATA_DIR}/final_run/figures'
IBTRACS_CSV = f'{DATA_DIR}/ibtracs.ALL.list.v04r01.csv'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

In [ ]:
save_prof = 1629339767 #8/19
# save_prof =  1630114503 #8/28


if save_prof ==1629339767:
    start_date=dt.datetime(2021,8,19)
    end_date=dt.datetime(2021,9,4)
    glider_location = f'{FORCING_DIR}/glider_location_close.csv'
    
elif save_prof == 1630114503:
    start_date = dt.datetime(2021,8,28)
    end_date = dt.datetime(2021,9,4)
    glider_location = f'{FORCING_DIR}/glider_location_082800.csv'

In [ ]:
def get_ndbc(bbox=None, time_start=None, time_end=None, buoy=None):
    '''
    Credit: Mike Smith
    Used to pull observational data from NDBC buoys off of ERDDAP
    '''
    bbox = bbox or [-100, -45, 5, 46]
    time_end = time_end or dt.date.today()
    time_start = time_start or (time_end - dt.timedelta(days=1))
    buoy = buoy or False
    time_formatter = '%Y-%m-%dT%H:%M:%SZ'

    e = ERDDAP(
        server='CSWC',
        protocol='tabledap',
        response='csv'
    )

    e.dataset_id = 'cwwcNDBCMet'
    e.constraints = {
        'time>=': time_start.strftime(time_formatter),
        'time<=': time_end.strftime(time_formatter),
    }

    if bbox:
        e.constraints['longitude>='] = bbox[0]
        e.constraints['longitude<='] = bbox[1]
        e.constraints['latitude>='] = bbox[2]
        e.constraints['latitude<='] = bbox[3]

    if buoy:
        e.constraints['station='] = buoy

    e.variables = [
        "latitude",
        "longitude",
        "wd",
        "wspd",
        "gst",
        "wvht",
        "dpd",
        "apd",
        "mwd",
        "bar",
        "atmp",
        "wtmp",
        "dewp",
        # "vis",
        # "ptdy",
        # "tide",
        "wspu",
        "wspv",
        "time",
    ]



    try:
        df = e.to_pandas(
            parse_dates=['time (UTC)'],
            skiprows=(1,)  # units information can be dropped.
        ).dropna()
    except HTTPError:
        df = pd.DataFrame()

    return df

In [ ]:
end_date = dt.datetime(2021,8,31)
buoydf = get_ndbc(bbox=None,time_start=start_date, time_end=end_date, buoy='42040')

In [ ]:
def wind_profile_law(u_ref,z_ref,z_new,alpha):
    
    u_new = u_ref * (z_new / z_ref)**alpha
    
    return u_new

In [ ]:
u_ref = buoydf['wspd (m s-1)'].values
z_ref = 4.1
z_new = 10
alpha = 0.11


u_new=wind_profile_law(u_ref,z_ref,z_new,alpha)

buoydf['wspd_10m']=u_new

# get u and v of 10m windspeed


D=buoydf['wd (degrees_true)'].values

rD = D*(np.pi/180)

ws = buoydf['wspd_10m'].values

buoydf['u_10m']=np.round(ws* -np.sin(rD),1)

buoydf['v_10m']=np.round(ws* -np.cos(rD),1)
# buoydf['u_10m'] = buoydf.ws_10m * np.cos(θ)
# buoydf['v_10m'] = buoydf.ws_10m * np.sin(θ)
buoydf

In [ ]:
def load_hrrr(start_date, end_date, buoy, point_location, height):
    directory=f'{FORCING_DIR}/hrrr_data_20210825_20210905_example/'
    sites = pd.read_csv(point_location, skipinitialspace=True)
    time_span_D = pd.date_range(start_date, end_date-timedelta(hours=23), freq='1D')  #  -timedelta(days=1) was removed from here
    hrrr_ws = []
    hrrr_EWwind = []
    hrrr_NWwind = []
    hrrr_dt = np.empty((0,), dtype='datetime64[m]')
    hrrr_height = []
    for ind, date in enumerate(time_span_D):
        file=''
        file = 'hrrr_data_' + date.strftime("%Y%m%d") + '.nc'
        
        # print(file)
        try:
            hrrr_ds=[]
            hrrr_ds = xr.open_dataset(Path(directory + file))
            # print(hrrr_ds)
            lats = hrrr_ds.gridlat_0.squeeze()
            lons = hrrr_ds.gridlon_0.squeeze()

            site_code = sites[sites['name'] == buoy].index[0]
            a = abs(lats - sites.latitude[site_code]) + abs(lons - sites.longitude[site_code])
            i, j = np.unravel_index(a.argmin(), a.shape)
            
            hrrr_ws = np.append(hrrr_ws, hrrr_ds.wind_speed[:, height, i, j])
            hrrr_dt = np.append(hrrr_dt, hrrr_ds.time)
            # print(hrrr_dt)
            hrrr_EWwind = np.append(hrrr_EWwind, hrrr_ds.eastward_wind[:, height, i, j])
            hrrr_NWwind = np.append(hrrr_NWwind, hrrr_ds.northward_wind[:, height, i, j])
            hrrr_height = np.append(hrrr_height, hrrr_ds.lv_HTGL1[height])

        except:
            time_span_H = pd.date_range(date, date + timedelta(hours=23), freq='H')
            hrrr_ws = np.append(hrrr_ws, np.empty(shape=(len(time_span_H))) * np.NAN)
            hrrr_dt = np.append(hrrr_dt, time_span_H)
            hrrr_EWwind = np.append(hrrr_EWwind, np.empty(shape=(len(time_span_H))) * np.NAN)
            hrrr_NWwind = np.append(hrrr_NWwind, np.empty(shape=(len(time_span_H))) * np.NAN)
            hrrr_height = np.append(hrrr_height, np.NAN)

    return hrrr_ws, hrrr_EWwind, hrrr_NWwind, hrrr_dt, hrrr_height

## Validate HRRR winds against buoy and glider-site observations

In [ ]:
buoy = 'LLNR' # buoy 42040

buoy_location = f'{FORCING_DIR}/buoy_location.csv'
height = 0 
hrrr_windSpeed, hrrr_EWwind, hrrr_NWwind, hrrr_time, hrrr_height = load_hrrr(start_date, end_date, buoy, buoy_location, height)

glider = 'ng645'
# other cases: glider_location_0826.csv, glider_location_0827.csv, glider_location_082811.csv, glider_location_082800.csv (all under FORCING_DIR)
height = 0 

ghrrr_windSpeed, ghrrr_EWwind, ghrrr_NWwind, ghrrr_time, ghrrr_height = load_hrrr(start_date, end_date, glider, glider_location, height)





# Make a big ol' plot to compare the wind products
fig = plt.figure(figsize=(20,20), constrained_layout=True)
xticks=(pd.date_range(start_date, end_date-timedelta(hours=23), freq='1D'))
landfallTime = '2021-08-29T17:00:00'
# xlims = (mdates.datestr2num("2021-8-28 00:00:00"),mdates.datestr2num(landfallTime))
xlims = (mdates.date2num(start_date),mdates.datestr2num("2021-09-04 00:00:00"))
spd_lims = (0,30)
dir_lims = (-26,26)
ICkw = {'ls':':', 'c':'red', 'alpha':0.5, 'lw':3}
ann_kwargs = {'fontsize':20, 'va':'bottom', 'ha':'right'}

# Break it up into sections
gs = fig.add_gridspec(3,1)

# Get colors for lines
N=3
colors = cmo.phase(np.linspace(0.6,0.95,N))

################################################################################
# Plot wind speed
ax = fig.add_subplot(gs[0,:])

ax.plot(buoydf['time (UTC)'],buoydf['wspd_10m'], c=colors[0], label='Buoy 42040', lw=3)
ax.plot(hrrr_time,hrrr_windSpeed, c=colors[1], label='HRRR 42040',lw=3)
ax.plot(ghrrr_time,ghrrr_windSpeed, c=colors[2], label='HRRR PWP Site',lw=3)
ax.legend(loc='upper right')
# ax.legend(loc='upper right')
ax.set_title('10m Wind Speed') #,fontweight='bold')

# Set the xticks and labels
ax.xaxis.set_major_formatter(myFmt)
ax.set_xticks(xticks)
ax.tick_params(labelbottom = False, bottom = False) # Remove bottom tick marks and labels
ax.set_xlim(xlims)
# ax.set_ylim(spd_lims)
ax.set_ylim(0,35)
ax.grid(True)
ax.set_ylabel('[ ms$^{-1}$ ]', labelpad=0)
ax.text(buoydf['time (UTC)'].iloc[0],32.2,'a)',fontweight='bold',fontsize=19)
# ax.annotate('(a)',(hrrr_time[-1],dir_lims[0]),xycoords='data', **ann_kwargs) 

################################################################################

################################################################################
# Plot U wind
ax = fig.add_subplot(gs[1,:])

ax.plot(buoydf['time (UTC)'],buoydf['u_10m'], c=colors[0], label='Buoy 42040', lw=3)
ax.plot(hrrr_time,hrrr_EWwind, c=colors[1], label='HRRR 42040',lw=3)
ax.plot(ghrrr_time,ghrrr_EWwind, c=colors[2], label='HRRR ng645',lw=3)

# ax.legend(loc='upper left')
ax.set_title('10m E-W Wind Speed') #,fontweight='bold')

# Set the xticks and labels
ax.xaxis.set_major_formatter(myFmt)
ax.set_xticks(xticks)
ax.tick_params(labelbottom = False, bottom = False) # Remove bottom tick marks and labels
ax.set_xlim(xlims)
ax.set_ylim(dir_lims)
ax.grid(True)
ax.set_ylabel('[ ms$^{-1}$ ]', labelpad=-15)
ax.text(buoydf['time (UTC)'].iloc[0],22,'b)',fontweight='bold',fontsize=19)
# ax.annotate('(b)',(hrrr_time[-1],dir_lims[0]),xycoords='data', **ann_kwargs) 

################################################################################

################################################################################
# Plot V wind
ax = fig.add_subplot(gs[2,:])

ax.plot(buoydf['time (UTC)'],buoydf['v_10m'], c=colors[0], label='Buoy 42040', lw=3)
ax.plot(hrrr_time,hrrr_NWwind, c=colors[1], label='HRRR 42040',lw=3)
ax.plot(ghrrr_time,ghrrr_NWwind, c=colors[2], label='HRRR PWP Site',lw=3)

# ax.legend()
ax.set_title('10m N-S Wind Speed') #,fontweight='bold')

# Set the xticks and labels
ax.xaxis.set_major_formatter(myFmt)
ax.set_xticks(xticks)
ax.set_xlim(xlims)
# ax.set_ylim(dir_lims)
ax.set_ylim(-30,35)
ax.grid(True)
ax.set_ylabel('[ ms$^{-1}$ ]', labelpad=-15)
ax.text(buoydf['time (UTC)'].iloc[0],30,'c)',fontweight='bold',fontsize=19)

# ax.annotate('(c)',(hrrr_time[-1],dir_lims[0]),xycoords='data', **ann_kwargs) 
################################################################################
plt.savefig(f'{FIGURES_DIR}/'+str(save_prof)+'wind_comparisons_at_10m_'+start_date.strftime('%m_%d')+'_0904.png',bbox_inches='tight')

# Grab forcing starting at the time you want to initialize the model from

In [ ]:
# Load the data back in with the right time range
glider = 'ng645'
close_prof_id= save_prof
# other cases: glider_location_0826.csv, glider_location_0827.csv, glider_location_082811.csv, glider_location_082800.csv (all under FORCING_DIR)
height = 0 

hrrr_windSpeed, hrrr_EWwind, hrrr_NWwind, hrrr_time, hrrr_height = load_hrrr(start_date, end_date, glider, glider_location, height)

# Need to make sure to start at the same time as the glider IC's (2021-08-28 12:16)
hrrr_tind = mdates.date2num(hrrr_time)>=mdates.date2num(start_date)
hrrr_windSpeed = hrrr_windSpeed[hrrr_tind]
hrrr_EWwind = hrrr_EWwind[hrrr_tind]
hrrr_NWwind = hrrr_NWwind[hrrr_tind]
hrrr_time = hrrr_time[hrrr_tind]

## HRRR
#Create dictionary with buoy forcing data
pwp_time = (mdates.date2num(hrrr_time) - mdates.date2num(hrrr_time[0])) # days since start
dummy = np.zeros_like(np.asarray(pwp_time))

# Compute Cd following (Jaimes et al, 2015)
Cd = (4-0.6*hrrr_windSpeed)*10**(-3)
midind = (hrrr_windSpeed >= 5) & (hrrr_windSpeed < 25)
Cd[midind] = (0.7375 + 0.0525*hrrr_windSpeed[midind])*10**(-3)
hiind = (hrrr_windSpeed >= 25)
Cd[hiind] = 2.05*10**(-3)

# Make time variable that starts at 1 and is in decimal days
# Flip signs of heat fluxes to be into ocean is positive
hrrratmo = {'time': {'dims': 'time', 'data': pwp_time}, 
        'sw': {'dims': 'time', 'data': dummy}, 
        'lw': {'dims': 'time', 'data': dummy}, 
        'qlat': {'dims': 'time', 'data': dummy}, 
        'qsens': {'dims': 'time', 'data': dummy}, 
        'tx': {'dims': 'time', 'data': 1.2 * Cd * hrrr_EWwind**2}, 
        'ty': {'dims': 'time', 'data': 1.2 * Cd * hrrr_NWwind**2}, 
        'precip': {'dims': 'time', 'data': dummy}}

hrrratmo_ds = xr.Dataset.from_dict(hrrratmo)
hrrratmo_ds.to_netcdf(f'{OUTPUT_DIR}/HRRR_Ida_windsOnly_'+str(save_prof)+'_'+start_date.strftime('%m_%d')+'HR_initime.nc')